<a href="https://colab.research.google.com/github/saisathwik2703/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saisathwik2703/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
the goal is to give each page an opportunity score and rank pages from the highest priority CTR opportunity to the lowest-priority opportunity.this is more useful than a simple yes/no rule because several pages can have different levels of opportunity and the team as limited review capacity. the output supports a content/SEO review queue:reviewer can start with the highest-ranked pages and decide what,if anything,should be changed.

In [5]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 299, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 299 (delta 130), reused 98 (delta 98), pack-reused 106 (from 1)
Receiving objects: 100% (299/299), 1.88 MiB | 4.16 MiB/s, done.
Resolving deltas: 100% (161/161), done.


In [9]:
# This cell is for CODE (numbers, a query, a check).
from pathlib import Path
import pandas as pd

DATA_PATH = Path(
    "/content/flyrank-ml-internship-starter/"
    "data/raw/content_refresh_anonymized.csv"
)

print("Dataset exists:", DATA_PATH.exists())

df = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Dataset exists: True
Loaded: /content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv
Shape: (30000, 44)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
a pages CTR opportunity score-how much the pages appears to under-capture clicks compared with the typical CTR for pages in the same position tier.
For this Week-2 framing exercise i use a simple observed proxy target:ctr_opportunity_proxy = 1 when a pages CTR is below the median CTR of its position tier,using pages with at least 100 impressions.this is only a starting proxy,not a claim that the page is objectively bad.
For a later production style model i would prefer a future-window outcome(for example,whether CTR improves after a review so the target represents an outcome that oocurs after the prediction.current CTR should not be used as a feature when it is also used to construct the target because that would create leakage.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Sketch the target/proxy column.
required = ["ctr", "impressions_90d", "position_tier"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

lane_df = df[df["impressions_90d"] >= 100].copy()
tier_median_ctr = lane_df.groupby("position_tier")["ctr"].transform("median")
lane_df["tier_median_ctr"] = tier_median_ctr
lane_df["ctr_gap_proxy"] = (lane_df["tier_median_ctr"] - lane_df["ctr"]).clip(lower=0)
lane_df["ctr_opportunity_proxy"] = (lane_df["ctr"] < lane_df["tier_median_ctr"]).astype(int)

print("Proxy target distribution:")
print(lane_df["ctr_opportunity_proxy"].value_counts().rename({0: "not_below_tier_median", 1: "below_tier_median"}))
print("\nExample target columns:")
print(lane_df[["position_tier", "ctr", "tier_median_ctr", "ctr_gap_proxy", "ctr_opportunity_proxy"]].head(10))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Proxy target distribution:
ctr_opportunity_proxy
not_below_tier_median    11699
below_tier_median        10307
Name: count, dtype: int64

Example target columns:
   position_tier   ctr  tier_median_ctr  ctr_gap_proxy  ctr_opportunity_proxy
0       striking  0.76             0.15           0.00                      0
1       page_3_5  0.05             0.06           0.01                      1
2       page_3_5  0.09             0.06           0.00                      0
3         page_1  0.49             0.23           0.00                      0
4       page_3_5  0.13             0.06           0.00                      0
5         page_1  0.03             0.23           0.20                      1
7       page_3_5  0.06             0.06           0.00                      0
8       page_3_5  0.09             0.06           0.00                      0
9         page_1  0.16             0.23           0.07                      1
10         top_3  1.55             0.19           0.00    

## 3. Success metric

*One metric you can defend. What number means 'good'?*
precision@50 means:among the 50 pages that the system ranks as the highest-priority opportunities,what fraction actually belong to defined opportunity group in the validation data.
A higher precision@50 is better because the real action is a limited review queue.if reviewrs can inspect about 50 pages first,the model should put a many genuinely high-priority pages as possible in those first 50 positions.would compare the model against a simple fixed rule or basline.A useful model should improve the quality of the top-ranked review queue without relying on leaked information.

In [11]:
# This cell is for CODE (numbers, a query, a check).
def precision_at_k(y_true, ranked_scores, k=50):
    """Fraction of the top-k ranked rows that are positive."""
    order = pd.Series(ranked_scores).sort_values(ascending=False).index[:k]
    return pd.Series(y_true).iloc[order].mean()

print("Success metric: Precision@50")
print("Good means: a larger share of the top 50 recommendations are true opportunities.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Success metric: Precision@50
Good means: a larger share of the top 50 recommendations are true opportunities.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
the page is the object that recieves the score and enters the review queue. the data contains pages level SEO/search measurements such as CTR,impressions,position tier,content characteristics,and trend-related fields.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Show the actual page-level slice used for this lane.
display_cols = [c for c in [
    "page_id", "client_id", "position_tier", "ctr", "impressions_90d",
    "avg_position", "content_type", "content_age_days", "word_count",
    "ctr_gap_proxy", "ctr_opportunity_proxy"
] if c in lane_df.columns]

print(f"One row = one page observation. Rows in lane slice: {len(lane_df):,}")
display(lane_df[display_cols].head(10))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


One row = one page observation. Rows in lane slice: 22,006


,client_id,position_tier,ctr,impressions_90d,avg_position,content_type,content_age_days,word_count,ctr_gap_proxy,ctr_opportunity_proxy
0,client_f369cb89fc,striking,0.76,3803,10.6,keyword article,187,3221.0,0.00,0
1,client_4e07408562,page_3_5,0.05,15320,20.3,keyword article,445,2481.0,0.01,1
2,client_7f2253d7e2,page_3_5,0.09,12581,36.5,keyword article,141,3515.0,0.00,0
3,client_19581e27de,page_1,0.49,11751,6.2,keyword article,463,NaN,0.00,0
4,client_3fdba35f04,page_3_5,0.13,19140,44.0,keyword article,263,2803.0,0.00,0
5,client_f369cb89fc,page_1,0.03,3970,8.5,keyword article,147,3080.0,0.20,1
7,client_19581e27de,page_3_5,0.06,1724,21.2,keyword article,445,NaN,0.00,0
8,client_6208ef0f77,page_3_5,0.09,32574,46.0,keyword article,90,3807.0,0.00,0
9,client_19581e27de,page_1,0.16,1240,4.9,keyword article,257,NaN,0.07,1
10,client_19581e27de,top_3,1.55,20919,2.2,keyword article,329,NaN,0.00,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
A fixed rule could say something like:"flag every page below the overall CTR average.'that is too coarse because CTR depends strongly on search postion, and pages also differ in content and other measurable characteristics.a single threshold can therefore create a noisy review queue.A scoring model can combine multiple non-leaking signals andproduce a ranked list instead of one hard cutoff.it can learn combinations that are difficult to express with a few hand written if statements, then be evaluated on held-out data.
the model still does not decide what a reviewer should change.it only prioritizes pages for human investigation.that makes this an ml decision-support problem rather than an automatic content-change system.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# A small baseline illustration: one fixed rule based only on position tier.
# This is a comparison idea, not the final model.
tier_rates = lane_df.groupby("position_tier")["ctr_opportunity_proxy"].mean().sort_values(ascending=False)
print("Opportunity rate by position tier:")
print(tier_rates)
print("\nThis shows why a position-aware comparison is more sensible than one global CTR threshold.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Opportunity rate by position tier:
position_tier
top_3       0.495310
page_1      0.493687
striking    0.485685
page_3_5    0.481017
deep        0.000000
Name: ctr_opportunity_proxy, dtype: float64

This shows why a position-aware comparison is more sensible than one global CTR threshold.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.